# Data Exploration

This notebook analyzes the `When2Call` SFT and DPO training splits with simple, transparent heuristics.

What it reports:
- heuristic class counts for SFT assistant targets
- heuristic class counts for DPO `chosen_response` and `rejected_response`
- tool-call / cannot-answer / request-for-info marker counts
- DPO chosen-vs-rejected class-pair breakdowns

Important note: these are heuristic labels, not official dataset labels.


In [4]:
from __future__ import annotations

import re
from collections import Counter
from pathlib import Path

import pandas as pd
from datasets import load_dataset, load_from_disk
from IPython.display import display

pd.set_option('display.max_colwidth', 140)

DATASET_NAME = 'nvidia/When2Call'
ALLOW_HF_FALLBACK = False


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'local_datasets_train_test').exists() or (candidate / 'local_datasets_test').exists():
            return candidate
    raise FileNotFoundError(
        'Could not find the repo root from the current working directory. '
        'Open the notebook from the Tool-Call-Decision-Making repo, or set Path.cwd() there first.'
    )


ROOT = find_repo_root(Path.cwd())
LOCAL_TRAIN_ROOT = ROOT / 'local_datasets_train_test'
LOCAL_TEST_ROOT = ROOT / 'local_datasets_test'
SFT_CONFIG = 'train_sft'
DPO_CONFIG = 'train_pref'

print(f'Repo root: {ROOT}')


Repo root: /Users/anirudhporuri/Tool-Call-Decision-Making


In [5]:
def safe_slug(*parts: str) -> str:
    return '__'.join(part.replace('/', '__') for part in parts if part)


def load_when2call_split(config: str, split: str = 'train', allow_hf_fallback: bool = ALLOW_HF_FALLBACK):
    local_dir = LOCAL_TRAIN_ROOT / safe_slug(DATASET_NAME, config) / split
    if local_dir.exists():
        print(f'Loading local snapshot: {local_dir}')
        return load_from_disk(str(local_dir))

    if not allow_hf_fallback:
        raise FileNotFoundError(
            f'Local dataset snapshot not found: {local_dir}. '
            'This notebook is set to avoid network fallback by default because that can look like it is hanging.'
        )

    print(f'Loading from Hugging Face: {DATASET_NAME} / {config} / {split}')
    return load_dataset(DATASET_NAME, config)[split]


sft_ds = load_when2call_split(SFT_CONFIG)
dpo_ds = load_when2call_split(DPO_CONFIG)

print(f'SFT rows: {len(sft_ds):,}')
print(f'DPO rows: {len(dpo_ds):,}')
print('\nSFT schema:')
print(sft_ds.features)
print('\nDPO schema:')
print(dpo_ds.features)


Loading local snapshot: /Users/anirudhporuri/Tool-Call-Decision-Making/local_datasets_train_test/nvidia__When2Call__train_sft/train
Loading local snapshot: /Users/anirudhporuri/Tool-Call-Decision-Making/local_datasets_train_test/nvidia__When2Call__train_pref/train
SFT rows: 15,000
DPO rows: 9,000

SFT schema:
{'tools': List(Value('string')), 'messages': List({'role': Value('string'), 'content': Value('string')})}

DPO schema:
{'tools': List(Value('string')), 'messages': List({'role': Value('string'), 'content': Value('string')}), 'chosen_response': {'role': Value('string'), 'content': Value('string')}, 'rejected_response': {'role': Value('string'), 'content': Value('string')}}


In [6]:
TOOLCALL_RE = re.compile(r'<TOOLCALL>(.*?)</TOOLCALL>', re.DOTALL)
CANNOT_PATTERNS = [
    r'\bsorry\b',
    r'\bapologies\b',
    r'\bapologize\b',
    r'\bapologise\b',
    r'\bcannot\b',
    r"\bcan't\b",
    r'\bunable\b',
]
REQUEST_PATTERNS = [
    r'\bcould you\b',
    r'\bcan you\b',
    r'\bplease provide\b',
    r'\bplease specify\b',
    r'\bwhat is\b',
    r"\bwhat's\b",
    r'\bwhich\b',
    r'\bto assist you better\b',
    r'\bjust to confirm\b',
    r'\bdo you have\b',
    r'\bmay i have\b',
]


def extract_toolcall_payload(text: str) -> str:
    if not isinstance(text, str):
        return ''
    match = TOOLCALL_RE.search(text)
    if match:
        return match.group(1).strip()
    return text.strip()


def has_tool_call_marker(text: str) -> bool:
    raw = text if isinstance(text, str) else ''
    payload = extract_toolcall_payload(raw)
    return ('<TOOLCALL>' in raw) or payload.startswith('{') or payload.startswith('[')


def has_cannot_answer_terms(text: str) -> bool:
    lower = (text or '').lower()
    return any(re.search(pattern, lower) for pattern in CANNOT_PATTERNS)


def looks_like_request_for_info(text: str) -> bool:
    lower = (text or '').lower().strip()
    if has_tool_call_marker(lower) or has_cannot_answer_terms(lower):
        return False
    return lower.endswith('?') or any(re.search(pattern, lower) for pattern in REQUEST_PATTERNS)


def heuristic_flags(text: str) -> dict[str, bool]:
    return {
        'tool_call_marker': has_tool_call_marker(text),
        'cannot_answer_terms': has_cannot_answer_terms(text),
        'request_for_info_pattern': looks_like_request_for_info(text),
    }


def heuristic_class(text: str) -> str:
    flags = heuristic_flags(text)
    if flags['tool_call_marker']:
        return 'tool_call'
    if flags['cannot_answer_terms']:
        return 'cannot_answer'
    if flags['request_for_info_pattern']:
        return 'request_for_info'
    return 'other_plain_text'


In [7]:
def summarize_texts(texts, label: str):
    rows = []
    for text in texts:
        flags = heuristic_flags(text)
        rows.append({
            'text': text,
            **flags,
            'heuristic_class': heuristic_class(text),
        })

    df = pd.DataFrame(rows)
    total = len(df)

    class_table = (
        df['heuristic_class']
        .value_counts(dropna=False)
        .rename_axis('heuristic_class')
        .reset_index(name='count')
    )
    class_table['percent'] = (100 * class_table['count'] / total).round(2)

    flag_table = pd.DataFrame(
        [
            {'signal': 'tool_call_marker', 'count': int(df['tool_call_marker'].sum())},
            {'signal': 'cannot_answer_terms', 'count': int(df['cannot_answer_terms'].sum())},
            {'signal': 'request_for_info_pattern', 'count': int(df['request_for_info_pattern'].sum())},
        ]
    )
    flag_table['percent'] = (100 * flag_table['count'] / total).round(2)

    print(f'===== {label} =====')
    print(f'n = {total:,}')
    display(class_table)
    display(flag_table)
    return df, class_table, flag_table


def print_examples(df: pd.DataFrame, label: str, n: int = 3):
    print(f'===== Sample examples: {label} =====')
    for cls in ['tool_call', 'cannot_answer', 'request_for_info', 'other_plain_text']:
        print(f'\n[{cls}]')
        samples = df.loc[df['heuristic_class'] == cls, 'text'].head(n).tolist()
        if not samples:
            print('  <none>')
            continue
        for i, sample in enumerate(samples, start=1):
            compact = sample.replace('\n', ' ')
            print(f'  {i}. {compact[:220]}')


In [8]:
sft_texts = [row['messages'][-1]['content'] for row in sft_ds]
sft_df, sft_class_table, sft_flag_table = summarize_texts(sft_texts, 'SFT assistant targets')
print_examples(sft_df, 'SFT assistant targets')


===== SFT assistant targets =====
n = 15,000


,heuristic_class,count,percent
0,cannot_answer,7571,50.47
1,request_for_info,7397,49.31
2,other_plain_text,32,0.21


,signal,count,percent
0,tool_call_marker,0,0.00
1,cannot_answer_terms,7571,50.47
2,request_for_info_pattern,7397,49.31


===== Sample examples: SFT assistant targets =====

[tool_call]
  <none>

[cannot_answer]
  1. Apologies, but I'm unable to provide real-time information or perform web searches. You may want to check a reliable news source for that.
  2. Apologies, but I'm unable to create a Facebook ad copy. I can provide suggestions or guidelines, but the actual creation would require human input and creativity.
  3. Apologies, I'm unable to provide that information as I don't have real-time access to vehicle databases. Please consult a reliable automotive resource for accurate details.

[request_for_info]
  1. To calculate the standard deviation, I'll need the specific numbers in your dataset. Could you please provide them?
  2. To provide the auto-complete suggestions, could you please specify the phrase or term you'd like me to use?
  3. To help you find relevant blogs, could you please specify a keyword related to 'blockchain technology' that I can use for the search?

[other_plain_text]
  1. To

In [9]:
dpo_chosen_texts = [row['chosen_response']['content'] for row in dpo_ds]
dpo_rejected_texts = [row['rejected_response']['content'] for row in dpo_ds]

dpo_chosen_df, dpo_chosen_class_table, dpo_chosen_flag_table = summarize_texts(
    dpo_chosen_texts,
    'DPO chosen responses',
)
print_examples(dpo_chosen_df, 'DPO chosen responses')

dpo_rejected_df, dpo_rejected_class_table, dpo_rejected_flag_table = summarize_texts(
    dpo_rejected_texts,
    'DPO rejected responses',
)
print_examples(dpo_rejected_df, 'DPO rejected responses')


===== DPO chosen responses =====
n = 9,000


,heuristic_class,count,percent
0,cannot_answer,3010,33.44
1,tool_call,3000,33.33
2,request_for_info,2981,33.12
3,other_plain_text,9,0.10


,signal,count,percent
0,tool_call_marker,3000,33.33
1,cannot_answer_terms,3010,33.44
2,request_for_info_pattern,2983,33.14


===== Sample examples: DPO chosen responses =====

[tool_call]
  1. <TOOLCALL>[{"name": "get_ico_calendar", "arguments": {"category": "_ico_cat_ecomm,_ico_cat_finance", "time_utc_offset": 28800, "tabname": "completed", "sort": "funds_raised"}}]</TOOLCALL>
  2. <TOOLCALL>[{"name": "rgb_to_cmyk", "arguments": {"r": 255, "g": 0, "b": 0}}]</TOOLCALL>
  3. <TOOLCALL>[{"name": "whole_foods_order", "arguments": {"store_location": "Chicago", "item_list": ["milk", "eggs"], "order_size": "small"}}]</TOOLCALL>

[cannot_answer]
  1. Apologies, but I'm unable to calculate the 99% confidence interval. This task requires statistical analysis that I currently cannot perform.
  2. Apologies, but I'm unable to perform calculations or identify specific numbers in a sequence.
  3. I apologize, but I'm unable to perform that task as I don't have the capability to access or retrieve specific data from a catalog.

[request_for_info]
  1. To assist you better, could you please specify the page number for the 

,heuristic_class,count,percent
0,other_plain_text,2920,32.44
1,cannot_answer,2197,24.41
2,tool_call,2026,22.51
3,request_for_info,1857,20.63


,signal,count,percent
0,tool_call_marker,2026,22.51
1,cannot_answer_terms,2197,24.41
2,request_for_info_pattern,1859,20.66


===== Sample examples: DPO rejected responses =====

[tool_call]
  1. <TOOLCALL>[{"name": "get_channels", "arguments": {"keyword": "pop"}}]</TOOLCALL>
  2. <TOOLCALL>[{"name": "address_reverse_geocoding", "arguments": {"coord": "40.7128,-74.0060", "datum": "wgs84", "coord_unit": "degree"}}]</TOOLCALL>
  3. <TOOLCALL>[{"name": "get_company_by_domain", "arguments": {"domain": "www.apple.com"}}]</TOOLCALL>

[cannot_answer]
  1. Apologies, I'm unable to provide that information as I don't have the capability to access or process such specific user data.
  2. Apologies, I'm unable to provide driving directions as I can't perform that task. Please use a mapping service for accurate information.
  3. Apologies, but I'm unable to provide that information as I don't have the capability to access or display real estate transactions.

[request_for_info]
  1. Which language do you prefer for the response? The options are English (default), Chinese, or Russian.
  2. To calculate the 99% confidence 

In [10]:
overview = pd.DataFrame(
    [
        {
            'dataset': 'SFT',
            'field': 'assistant_target',
            'n_rows': len(sft_df),
            'tool_call_marker_count': int(sft_df['tool_call_marker'].sum()),
            'cannot_answer_count': int(sft_df['cannot_answer_terms'].sum()),
            'request_for_info_count': int(sft_df['request_for_info_pattern'].sum()),
        },
        {
            'dataset': 'DPO',
            'field': 'chosen_response',
            'n_rows': len(dpo_chosen_df),
            'tool_call_marker_count': int(dpo_chosen_df['tool_call_marker'].sum()),
            'cannot_answer_count': int(dpo_chosen_df['cannot_answer_terms'].sum()),
            'request_for_info_count': int(dpo_chosen_df['request_for_info_pattern'].sum()),
        },
        {
            'dataset': 'DPO',
            'field': 'rejected_response',
            'n_rows': len(dpo_rejected_df),
            'tool_call_marker_count': int(dpo_rejected_df['tool_call_marker'].sum()),
            'cannot_answer_count': int(dpo_rejected_df['cannot_answer_terms'].sum()),
            'request_for_info_count': int(dpo_rejected_df['request_for_info_pattern'].sum()),
        },
    ]
)

for col in ['tool_call_marker_count', 'cannot_answer_count', 'request_for_info_count']:
    pct_col = col.replace('_count', '_percent')
    overview[pct_col] = (100 * overview[col] / overview['n_rows']).round(2)

display(overview)


,dataset,field,n_rows,tool_call_marker_count,cannot_answer_count,request_for_info_count,tool_call_marker_percent,cannot_answer_percent,request_for_info_percent
0,SFT,assistant_target,15000,0,7571,7397,0.00,50.47,49.31
1,DPO,chosen_response,9000,3000,3010,2983,33.33,33.44,33.14
2,DPO,rejected_response,9000,2026,2197,1859,22.51,24.41,20.66


In [11]:
pair_df = pd.DataFrame(
    {
        'chosen_class': dpo_chosen_df['heuristic_class'],
        'rejected_class': dpo_rejected_df['heuristic_class'],
        'chosen_tool_call': dpo_chosen_df['tool_call_marker'],
        'rejected_tool_call': dpo_rejected_df['tool_call_marker'],
    }
)

pair_breakdown = (
    pair_df.value_counts(['chosen_class', 'rejected_class'])
    .rename('count')
    .reset_index()
)
pair_breakdown['percent'] = (100 * pair_breakdown['count'] / len(pair_df)).round(2)
display(pair_breakdown)

tool_pair_breakdown = (
    pair_df.value_counts(['chosen_tool_call', 'rejected_tool_call'])
    .rename('count')
    .reset_index()
)
tool_pair_breakdown['percent'] = (100 * tool_pair_breakdown['count'] / len(pair_df)).round(2)
display(tool_pair_breakdown)


,chosen_class,rejected_class,count,percent
0,tool_call,cannot_answer,1093,12.14
1,tool_call,other_plain_text,1038,11.53
2,request_for_info,cannot_answer,1030,11.44
3,cannot_answer,tool_call,1021,11.34
4,request_for_info,tool_call,998,11.09
5,cannot_answer,other_plain_text,981,10.90
6,cannot_answer,request_for_info,941,10.46
7,request_for_info,other_plain_text,900,10.00
8,tool_call,request_for_info,863,9.59
9,cannot_answer,cannot_answer,67,0.74


,chosen_tool_call,rejected_tool_call,count,percent
0,False,False,3980,44.22
1,True,False,2994,33.27
2,False,True,2020,22.44
3,True,True,6,0.07


## Balanced SFT Dataset Check

This section loads the generated balanced SFT JSONL and checks both:
- the declared `behavior_class` balance written by the dataset builder
- the heuristic class balance recovered from the final assistant targets


In [12]:
balanced_sft_path = ROOT / 'Data_Management' / 'generated_datasets' / 'when2call_balanced_sft_3x3000.jsonl'
if not balanced_sft_path.exists():
    raise FileNotFoundError(
        f'Balanced SFT dataset not found: {balanced_sft_path}. '
        'Run Data_Management/build_balanced_sft_dataset.py first.'
    )

balanced_sft_df = pd.read_json(balanced_sft_path, lines=True)
print(f'Balanced SFT path: {balanced_sft_path}')
print(f'Rows: {len(balanced_sft_df):,}')

declared_balance = (
    balanced_sft_df['behavior_class']
    .value_counts(dropna=False)
    .rename_axis('behavior_class')
    .reset_index(name='count')
)
declared_balance['percent'] = (100 * declared_balance['count'] / len(balanced_sft_df)).round(2)
display(declared_balance)

balanced_target_texts = [messages[-1]['content'] for messages in balanced_sft_df['messages']]
balanced_target_df, balanced_target_class_table, balanced_target_flag_table = summarize_texts(
    balanced_target_texts,
    'Balanced SFT assistant targets',
)

declared_vs_heuristic = pd.crosstab(
    balanced_sft_df['behavior_class'],
    balanced_target_df['heuristic_class'],
    rownames=['declared_behavior_class'],
    colnames=['heuristic_class'],
)
display(declared_vs_heuristic)


Balanced SFT path: /Users/anirudhporuri/Tool-Call-Decision-Making/Data_Management/generated_datasets/when2call_balanced_sft_3x3000.jsonl
Rows: 9,000


,behavior_class,count,percent
0,request_for_info,3000,33.33
1,cannot_answer,3000,33.33
2,tool_call,3000,33.33


===== Balanced SFT assistant targets =====
n = 9,000


,heuristic_class,count,percent
0,request_for_info,3000,33.33
1,cannot_answer,3000,33.33
2,tool_call,3000,33.33


,signal,count,percent
0,tool_call_marker,3000,33.33
1,cannot_answer_terms,3000,33.33
2,request_for_info_pattern,3002,33.36


heuristic_class,cannot_answer,request_for_info,tool_call
declared_behavior_class,,,
cannot_answer,3000,0,0
request_for_info,0,3000,0
tool_call,0,0,3000


## Test Set Label Distribution

The `test/mcq` split already includes the gold label in `correct_answer`, so we can inspect the class distribution directly.


In [13]:
def load_when2call_test_mcq():
    local_dir = ROOT / 'local_datasets_test' / safe_slug(DATASET_NAME, 'test') / 'mcq'
    if local_dir.exists():
        print(f'Loading local snapshot: {local_dir}')
        return load_from_disk(str(local_dir))

    print(f'Loading from Hugging Face: {DATASET_NAME} / test / mcq')
    return load_dataset(DATASET_NAME, 'test')['mcq']


test_ds = load_when2call_test_mcq()
print(f'Test rows: {len(test_ds):,}')
print('\nTest schema:')
print(test_ds.features)

test_label_distribution = (
    pd.Series(test_ds['correct_answer'])
    .value_counts(dropna=False)
    .rename_axis('correct_answer')
    .reset_index(name='count')
)
test_label_distribution['percent'] = (100 * test_label_distribution['count'] / len(test_ds)).round(2)
display(test_label_distribution)


Loading local snapshot: /Users/anirudhporuri/Tool-Call-Decision-Making/local_datasets_test/nvidia__When2Call__test/mcq
Test rows: 3,652

Test schema:
{'uuid': Value('string'), 'source': Value('string'), 'source_id': Value('string'), 'question': Value('string'), 'correct_answer': Value('string'), 'answers': {'direct': Value('string'), 'tool_call': Value('string'), 'request_for_info': Value('string'), 'cannot_answer': Value('string')}, 'target_tool': Value('string'), 'tools': List(Value('string')), 'orig_tools': List(Value('string')), 'orig_question': Value('string'), 'held_out_param': Value('string')}


,correct_answer,count,percent
0,cannot_answer,1295,35.46
1,tool_call,1295,35.46
2,request_for_info,1062,29.08
